#### Chronos 2 For Time Series With Fixed Data - Two Targets
Improves on an earlier notebook by building around fixed data: issues with event to place attribution have been resolved. Additional country-specific columns have also been added, as has election information. 

This version improves on the previous by have separate targets for local and regional conflict. 
-local conflict is defined just with the local variables LocalTotalFatalities or LocalAvgSeverity >= 3. 
-Regional conflict is definied as local conflict OR RegionalTotalFatalities or RegionalAvgSeverity >= 3. 

Builds a model with Amazon's Chronos-2 pretrained model.
Heavily adapted from https://huggingface.co/amazon/chronos-2 and https://colab.research.google.com/github/amazon-science/chronos-forecasting/blob/main/notebooks/chronos-2-quickstart.ipynb#scrollTo=39de5d7e with some reference to https://deepwiki.com/amazon-science/chronos-forecasting/1-overview

Chronos 2 was selected as it supports cross training. Serious difficulties overcome with Chronos 1 and Chronos Bolt. Previous attempt with TimeSeriesTransformerForPrediction also failed as couldn; get the model to make predictions despite being able to train it. That model was selected for ONNX export potential (ease of use from within C#).

As per Amazon's HuggingFace model page: "Chronos-2 is a 120M-parameter, encoder-only time series foundation model for zero-shot forecasting." The encoder learns representations of the past. As a foundation model, it is trained on a large number of time series datasets across different domains, therefore should generalise out to this problem. As a model with the ability to do zero-shot forecasting, it can make predictions without havign to be trained on our dataset, i.e, we are providing it context of the past data to allow a prediction, not adjusting internal model weights as per our dataset- making the predictions means it does not actually learn anything new, it is acting as block box data in -> maths -> predictions out.

Chronos 2 improves over prior Chronos models by adding Cross-learning across items, Multivariate Forecasting, Past-only (real/categorical) covariates, and better Known future (real/categorical) covariates support. It also has a larger context window of 8192 comapred to Chronos-Bolt (2048) and Chronos 1 (512) and support for much longer prediction steps (1024)

Without a decoder, Chronos 2 is not capable of generating new tokens, or their probabilities. Instead we get the quartiles (often 0.1, 0.5, 0.9).

## Load Dataset

In [17]:
import pandas as pd
df_context_full = pd.read_csv("Dataset Preprocessed\eastafrica3_2026-09-20_Train.csv")
df_future_full = pd.read_csv("Dataset Preprocessed\eastafrica3_2026-09-20_Test.csv")

### Combine

In [18]:
df = pd.concat([df_context_full, df_future_full], ignore_index=True)
del df_context_full
del df_future_full

### Minor Preprocessing

In [19]:
#format the perdiodStart as proper date:
df['periodStart'] = pd.to_datetime(df['periodStart'],  format="%Y-%m-%d")

In [20]:
for col in df.columns:
    print(col)

id
name
country
latitude
longitude
minBorderDistanceKm
minCapitalDistanceKm
periodStart
periodEnd
LocalEventCount
LocalDistinctActorCount
LocalDistinctEventTypes
LocalDistinctEventSubtypes
LocalTotalFatalities
LocalAvgSeverity
LocalProtestersEventCount
LocalStateForcesEventCount
LocalPoliticalMilitiaEventCount
LocalIdentityMilitiaEventCount
LocalRebelGroupEventCount
LocalRiotersEventCount
LocalCiviliansEventCount
LocalOtherTypeEventCount
RegionalEventCount
RegionalDistinctActorCount
RegionalDistinctEventTypes
RegionalDistinctEventSubTypes
RegionalTotalFatalities
RegionalAvgSeverity
RegionalProtestersEventCount
RegionalStateForcesEventCount
RegionalPoliticalMilitiaEventCount
RegionalIdentityMilitiaEventCount
RegionalCiviliansEventCount
RegionalRebelGroupEventCount
RegionalRiotersEventCount
RegionalOtherTypeEventCount
conflict_indicator
Population, total [SP.POP.TOTL]
Control of Corruption - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_CC_EST]
Rule of Law - Governance estimate (ap

### Applying replacement target variables

In [21]:
df = df.drop('conflict_indicator', axis=1)

In [22]:

#Context:
df['target_conflict_indicator_local'] = (
    (df['LocalTotalFatalities'] >= 3) |
    (df['LocalAvgSeverity'] >= 3)
).astype(int)

df['target_conflict_indicator_regional'] = (
    ##(df_context_full['target_conflict_indicator_local'] >= 1) |
    (df['RegionalTotalFatalities'] >= 3) |
    (df['RegionalAvgSeverity'] >= 3)
).astype(int)




In [23]:

#remove targets and unknowable values from future covariates (we'll remove the target from the subsamples when we create them). 
df_future_full_columns = ["id", "name",	"country",	"latitude",	"longitude",	"minBorderDistanceKm",	"minCapitalDistanceKm",	"periodStart", 'is_voting', 'target_conflict_indicator_local', 'target_conflict_indicator_regional']


## Validation

In [25]:
#full:
context_counts = (
    df.groupby("id")
      .size()
      .reset_index(name="count")
)

bad_ids_context = context_counts.query("count != 139")

print("Bad IDs in df_future_full:", bad_ids_context["id"].tolist())

# Show counts for each bad ID
print("Counts for bad IDs (df):")
print(bad_ids_context)





Bad IDs in df_future_full: ['Kenya_Busia', 'Kenya_Gatare', 'Kenya_Mbale', 'Uganda_Kigoma', 'Uganda_Rubirizi']
Counts for bad IDs (df):
                    id  count
3468       Kenya_Busia    417
3695      Kenya_Gatare    417
4808       Kenya_Mbale    417
12782    Uganda_Kigoma    417
13463  Uganda_Rubirizi    417


In [28]:
#5x instances are duplicated 3x. Possibly from re-running cells in Preprocessing.ipynb. Just delete for now: 
df = df[~df["id"].isin(bad_ids_context["id"].tolist())]


## Preselecting contextual data for class balance

### Overall Class Breakdown: Raw observations

#### Local

In [29]:
breakdown = (
    df['target_conflict_indicator_local']
      .value_counts()
      .sort_index()
      .rename_axis('target_conflict_indicator_local')
      .reset_index(name='count')
)

# Add percentage column
total = len(df)
breakdown['percentage'] = (breakdown['count'] / total * 100).round(2)

print(breakdown)


   target_conflict_indicator_local    count  percentage
0                                0  1846949       98.03
1                                1    37196        1.97


#### Regional

In [31]:
breakdown = (
    df['target_conflict_indicator_regional']
      .value_counts()
      .sort_index()
      .rename_axis('target_conflict_indicator_regional')
      .reset_index(name='count')
)

# Add percentage column
total = len(df)
breakdown['percentage'] = (breakdown['count'] / total * 100).round(2)

print(breakdown)

   target_conflict_indicator_regional    count  percentage
0                                   0  1881777       99.87
1                                   1     2368        0.13


### Id Breakdown by Maximum conflict_indicator

In [32]:
id_max = (
    df.groupby('id')['target_conflict_indicator_local']
      .max()
      .reset_index()
)

breakdown = (
    id_max['target_conflict_indicator_local']
        .value_counts()
        .sort_index()
        .rename_axis('target_conflict_indicator_local')
        .reset_index(name='count')
)

total_ids = id_max.shape[0]
breakdown['percentage'] = (breakdown['count'] / total_ids * 100).round(2)

print(breakdown)

   target_conflict_indicator_local  count  percentage
0                                0   4882       36.02
1                                1   8673       63.98


In [33]:
id_max = (
    df.groupby('id')['target_conflict_indicator_regional']
      .max()
      .reset_index()
)

breakdown = (
    id_max['target_conflict_indicator_regional']
        .value_counts()
        .sort_index()
        .rename_axis('target_conflict_indicator_regional')
        .reset_index(name='count')
)

total_ids = id_max.shape[0]
breakdown['percentage'] = (breakdown['count'] / total_ids * 100).round(2)

print(breakdown)

   target_conflict_indicator_regional  count  percentage
0                                   0  13303       98.14
1                                   1    252        1.86


In [177]:
df.to_parquet("Dataset Preprocessed/lora-full-dataset.parquet")

In [137]:
import pandas as pd
from datetime import datetime

def build_context_dataset(
    df_source: pd.DataFrame,
    cutoffDate: datetime | None = None,
    ids: str | None = None,
    n_high: int = 50,
    n_low: int = 25,
    n_spike: int = 25,
    n_transition: int = 25,
):

    df_source = df_source.copy()

    specific_ids = []
    
    if ids is not None:
        if isinstance(ids, list):
            # Normalize each ID in the list
            specific_ids = [
                x.strip().replace(" ", "_")
                for x in ids
                if isinstance(x, str) and x.strip() != ""
            ]
        else:
            raise TypeError("ids must be either a list of strings")

    
    # Use the new indicator columns as the conflict signal
    # Assuming they are 0/1 or boolean; cast to int to be safe.
    df_source["conflict_score"] = (
        df_source["target_conflict_indicator_local"].astype(int)
        + df_source["target_conflict_indicator_regional"].astype(int)
    )

    # Aggregate by id (place)
    agg = (
        df_source.groupby("id")
        .agg(
            total_conflict=("conflict_score", "sum"),
            max_spike=("conflict_score", lambda s: s.diff().abs().max()),
        )
        .reset_index()
    )

    # --- High conflict places -------------------------------------------------
    high_ids = (
        agg.sort_values("total_conflict", ascending=False)
        .head(n_high)["id"]
        .tolist()
    )

    # --- Low conflict places --------------------------------------------------
    low_ids = (
        agg.sort_values("total_conflict", ascending=True)
        .head(n_low)["id"]
        .tolist()
    )

    # --- Spike-based places ---------------------------------------------------
    spike_ids = (
        agg.sort_values("max_spike", ascending=False)
        .head(n_spike)["id"]
        .tolist()
    )

    #detect transitions: any change from previous timestep
    df_source["transition_flag"] = (
        df_source.groupby("id")["conflict_score"]
        .diff()
        .abs()
        > 0
    ).astype(int)
    
    #count transitions per id
    transition_ids = (
        df_source.groupby("id")["transition_flag"]
        .sum()
        .sort_values(ascending=False)
        .head(n_transition)
        .index
        .tolist()
    )
    
    # Combine unique ids
    selected_ids = (
        set(specific_ids)
        | set(high_ids)
        | set(low_ids)
        | set(spike_ids)
        | set(transition_ids)
    )

    #extract full windows for each selected id
    df_context = df_source[df_source["id"].isin(selected_ids)].copy().drop_duplicates()


    
    #dynamically build out the future set:
    df_future_columns =  ["id", "name",	"country",	"latitude",	"longitude",	"minBorderDistanceKm",	"minCapitalDistanceKm",	"periodStart", 'is_voting',  'target_conflict_indicator_local', 'target_conflict_indicator_regional'] #these are covariates we can know at the time of prediction. We also output the targets: we'll remove them when actually doing the training, but useful for evaluation.
    df_future = df_context[df_context['periodStart'] >= cutoffDate][df_future_columns].copy()


    #if we do specify a specific id(s), apply dummy values to the other ids in df_future:
    if specific_ids or len(ids) > 0:
        # IDs we do NOT care about
        other_ids = set(df_future["id"].unique()) - set(specific_ids)

        # Overwrite unwanted IDs with defaults
        for oid in other_ids:
            mask = df_future["id"] == oid

            # Overwrite all future covariates with safe defaults
            df_future.loc[mask, "name"] = None
            df_future.loc[mask, "country"] = None
            df_future.loc[mask, "latitude"] = 0.0
            df_future.loc[mask, "longitude"] = 0.0
            df_future.loc[mask, "minBorderDistanceKm"] = 0.0
            df_future.loc[mask, "minCapitalDistanceKm"] = 0.0
            df_future.loc[mask, "is_voting"] = False
    


    
    #now use cutoffDate to remove future records from the testing set:
    df_context = df_context[df_context['periodStart'] <= cutoffDate]   

    #validate: every id must have exactly 12 observations
    counts = df_future.groupby("id").size()
    invalid_ids = counts[counts < 12]
    if len(invalid_ids) > 0 or  df_future.empty:
        raise ValueError(f"Invalid future window: the following ids do not have 12 observations:\n{invalid_ids}. Try setting the cutoff date to an earlier value.")

    #remove extra future records:
    df_future = (
        df_future.groupby("id")
        .head(12)
        .reset_index(drop=True)
    )
    
    return df_context.sort_values(["id", "periodStart"]), df_future.sort_values(["id", "periodStart"])

In [144]:
#df_context, df_future = build_context_dataset(df_source = df, cutoffDate = '2023-08-01', ids=["Sudan_Omdurman_-_Al_Doha"] )
df_context, df_future = build_context_dataset(df_source = df, cutoffDate = '2023-08-01', ids=[])


#print(df_context)
print(df_context.groupby("id").size().reset_index(name="count"))
print("shape: ", df_context.shape)

#print(df_context)
print(df_future.groupby("id").size().reset_index(name="count"))
print("shape: ", df_future.shape)
print(sorted(df_future['id'].unique()))


                                   id  count
0    Central_African_Republic_Bambari    112
1     Central_African_Republic_Bozoum    112
2       Central_African_Republic_Bria    112
3                      Egypt_Al_Arish    112
4                         Egypt_Rafah    112
..                                ...    ...
108                     Uganda_Kitgum    112
109                     Uganda_Kiziba    112
110                     Uganda_Koboko    112
111                     Uganda_Kotido    112
112                      Uganda_Zombo    112

[113 rows x 2 columns]
shape:  (12656, 71)
                                   id  count
0    Central_African_Republic_Bambari     12
1     Central_African_Republic_Bozoum     12
2       Central_African_Republic_Bria     12
3                      Egypt_Al_Arish     12
4                         Egypt_Rafah     12
..                                ...    ...
108                     Uganda_Kitgum     12
109                     Uganda_Kiziba     12
110        

## Vanilla Model

In [139]:
#Install the amazon chronos package for chronos 2
!pip install chronos-forecasting

from chronos import Chronos2Pipeline

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional: restrict to one GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"



vanilla_chronos2_pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu"   #"CUDA also an option if you're a baller with a big GPU (I am not). 
)


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

### Local conflict (Vanilla)

In [145]:
# Generate predictions with covariates: takes roughly 4 mins. 
import datetime;
import joblib

ct = datetime.datetime.now()
print("current time:", ct)


pred_df_local = vanilla_chronos2_pipeline.predict_df(
    df =df_context,
    future_df=df_future.drop(columns=['target_conflict_indicator_local', 'target_conflict_indicator_regional']).copy(), #drop the target cols from the future df; we don't need them. 
    prediction_length=12,  # Number of steps to forecast
    quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
    id_column="id",  # Column identifying different time series
    timestamp_column="periodStart",  # Column with datetime information
    target="target_conflict_indicator_local",  # Column(s) with time series values to predict
    cross_learning=True,  # Enable cross-learning;
)
pred_df_local

ct = datetime.datetime.now()
print("current time:", ct)

joblib.dump(vanilla_chronos2_pipeline, f"chronos2_model_local_vanilla.pkl") #save the full model as a pickle file. 

current time: 2026-09-27 13:55:15.729863
current time: 2026-09-27 13:56:48.459265


['chronos2_model_local_vanilla.pkl']

### Regional Conflict (Vanilla)

In [146]:
# Generate predictions with covariates: takes roughly 4 mins. 

import datetime;

ct = datetime.datetime.now()
print("current time:", ct)


pred_df_regional = vanilla_chronos2_pipeline.predict_df(
    df =df_context,
    future_df=df_future.drop(columns=['target_conflict_indicator_local', 'target_conflict_indicator_regional']).copy(),  #drop the target cols from the future df; we don't need them. 
    prediction_length=12,  # Number of steps to forecast 
    quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
    id_column="id",  # Column identifying different time series
    timestamp_column="periodStart",  # Column with datetime information
    target="target_conflict_indicator_regional",  # Column(s) with time series values to predict
    cross_learning=True,  # Enable cross-learning;
)
pred_df_regional

ct = datetime.datetime.now()
print("current time:", ct)

joblib.dump(vanilla_chronos2_pipeline, f"chronos2_model_regional_vanilla.pkl") #save the full model as a pickle file. 

current time: 2026-09-27 13:56:49.169424
current time: 2026-09-27 13:58:39.693551


['chronos2_model_regional_vanilla.pkl']

### Making predictions & Evaluating

In [147]:
eval_df_local = pred_df_local.merge(
    df_future[["id", "periodStart", "target_conflict_indicator_local"]],
    on=["id", "periodStart"],
    how="inner"
)
eval_df_local

eval_df_regional = pred_df_regional.merge(
    df_future[["id", "periodStart", "target_conflict_indicator_regional"]],
    on=["id", "periodStart"],
    how="inner"
)
eval_df_regional

,id,periodStart,target_name,predictions,0.1,0.5,0.9,target_conflict_indicator_regional
0,Central_African_Republic_Bambari,2023-08-03,target_conflict_indicator_regional,0.009988,-0.008073,0.009988,1.000476,0
1,Central_African_Republic_Bambari,2023-08-31,target_conflict_indicator_regional,0.012404,-0.010543,0.012404,1.054102,0
2,Central_African_Republic_Bambari,2023-09-28,target_conflict_indicator_regional,0.002286,-0.008750,0.002286,0.968052,0
3,Central_African_Republic_Bambari,2023-10-26,target_conflict_indicator_regional,0.018730,-0.011480,0.018730,1.087256,0
4,Central_African_Republic_Bambari,2023-11-23,target_conflict_indicator_regional,0.007150,-0.009012,0.007150,1.057615,0
...,...,...,...,...,...,...,...,...
1351,Uganda_Zombo,2024-02-15,target_conflict_indicator_regional,0.000081,-0.000440,0.000081,0.001021,0
1352,Uganda_Zombo,2024-03-14,target_conflict_indicator_regional,0.000106,-0.000351,0.000106,0.000808,0
1353,Uganda_Zombo,2024-04-11,target_conflict_indicator_regional,0.000075,-0.000402,0.000075,0.000845,0
1354,Uganda_Zombo,2024-05-09,target_conflict_indicator_regional,0.000012,-0.000611,0.000012,0.000568,0


### Evaluation: Local Vanilla

* Recall is  TP / (TP + FN)
* Precision is TP / (TP+FP)
* F-1 score is the harmonic mean of recall and precision: it balances false negatives against false positives equally.
* F-2 is weights recall twice as important as precision.
* F-0.5 weights precision twice as much as recall. 

In [156]:
import numpy as np
from sklearn.metrics import (
    precision_recall_curve, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_auc_score
)

# ============================
# 1. Extract true labels + probabilities
# ============================

y_true = eval_df_local["target_conflict_indicator_local"].values          # binary 0/1 target
y_prob = eval_df_local["0.5"].values                       # Chronos2 median probability


# ============================
# 2. Compute precision-recall curve
# ============================

precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

# Align lengths: thresholds is length N-1
precision = precision[:-1]
recall = recall[:-1]


# ============================
# 3. Compute F0.5, F1, F2 scores
# ============================

# F0.5: precision weighted twice as much as recall
f05_scores = (1.25 * precision * recall) / (0.25 * precision + recall + 1e-9)

# F1: balanced
f1_scores_curve = (2 * precision * recall) / (precision + recall + 1e-9)

# F2: recall weighted twice as much as precision
f2_scores = (5 * precision * recall) / (4 * precision + recall + 1e-9)


# ============================
# 4. Select best thresholds
# ============================

best_idx_f05 = np.argmax(f05_scores)
best_threshold_f05 = thresholds[best_idx_f05]

best_idx_f1 = np.argmax(f1_scores_curve)
best_threshold_f1 = thresholds[best_idx_f1]

best_idx_f2 = np.argmax(f2_scores)
best_threshold_f2 = thresholds[best_idx_f2]


print("Best threshold (F0.5):", best_threshold_f05)
print("Best threshold (F1):", best_threshold_f1)
print("Best threshold (F2):", best_threshold_f2)


# ============================
# 5. Function to compute metrics for a given threshold
# ============================

def compute_metrics(threshold, name):
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n===== Metrics for {name} (threshold={threshold:.6f}) =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1:", f1_score(y_true, y_pred))
    print("ROC AUC:", roc_auc_score(y_true, y_prob))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


# ============================
# 6. Output metrics for each threshold
# ============================

compute_metrics(best_threshold_f05, "F0.5 (precision-weighted)")
compute_metrics(best_threshold_f1, "F1 (balanced)")
compute_metrics(best_threshold_f2, "F2 (recall-weighted)")


Best threshold (F0.5): 0.9697267
Best threshold (F1): 0.14921579
Best threshold (F2): 0.005777836

===== Metrics for F0.5 (precision-weighted) (threshold=0.969727) =====
Accuracy: 0.81047197640118
Precision: 0.8366013071895425
Recall: 0.5529157667386609
F1: 0.6657997399219766
ROC AUC: 0.8961855951859796
Confusion Matrix:
[[843  50]
 [207 256]]

===== Metrics for F1 (balanced) (threshold=0.149216) =====
Accuracy: 0.8178466076696165
Precision: 0.6824324324324325
Recall: 0.8725701943844493
F1: 0.7658767772511849
ROC AUC: 0.8961855951859796
Confusion Matrix:
[[705 188]
 [ 59 404]]

===== Metrics for F2 (recall-weighted) (threshold=0.005778) =====
Accuracy: 0.7529498525073747
Precision: 0.5844327176781002
Recall: 0.9568034557235421
F1: 0.7256347256347256
ROC AUC: 0.8961855951859796
Confusion Matrix:
[[578 315]
 [ 20 443]]


### Evaluation: Vanilla Regional

In [158]:

import numpy as np
from sklearn.metrics import (
    precision_recall_curve, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_auc_score
)

# ============================
# 1. Extract true labels + probabilities
# ============================

y_true = eval_df_regional["target_conflict_indicator_regional"].values          # binary 0/1 target
y_prob = eval_df_regional["0.5"].values                       # Chronos2 median probability


# ============================
# 2. Compute precision-recall curve
# ============================

precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

# Align lengths: thresholds is length N-1
precision = precision[:-1]
recall = recall[:-1]


# ============================
# 3. Compute F0.5, F1, F2 scores
# ============================

# F0.5: precision weighted twice as much as recall
f05_scores = (1.25 * precision * recall) / (0.25 * precision + recall + 1e-9)

# F1: balanced
f1_scores_curve = (2 * precision * recall) / (precision + recall + 1e-9)

# F2: recall weighted twice as much as precision
f2_scores = (5 * precision * recall) / (4 * precision + recall + 1e-9)


# ============================
# 4. Select best thresholds
# ============================

best_idx_f05 = np.argmax(f05_scores)
best_threshold_f05 = thresholds[best_idx_f05]

best_idx_f1 = np.argmax(f1_scores_curve)
best_threshold_f1 = thresholds[best_idx_f1]

best_idx_f2 = np.argmax(f2_scores)
best_threshold_f2 = thresholds[best_idx_f2]


print("Best threshold (F0.5):", best_threshold_f05)
print("Best threshold (F1):", best_threshold_f1)
print("Best threshold (F2):", best_threshold_f2)


# ============================
# 5. Function to compute metrics for a given threshold
# ============================

def compute_metrics(threshold, name):
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n===== Metrics for {name} (threshold={threshold:.6f}) =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1:", f1_score(y_true, y_pred))
    print("ROC AUC:", roc_auc_score(y_true, y_prob))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


# ============================
# 6. Output metrics for each threshold
# ============================

compute_metrics(best_threshold_f05, "F0.5 (precision-weighted)")
compute_metrics(best_threshold_f1, "F1 (balanced)")
compute_metrics(best_threshold_f2, "F2 (recall-weighted)")


Best threshold (F0.5): 0.33660486
Best threshold (F1): 0.30767944
Best threshold (F2): 0.0007393388

===== Metrics for F0.5 (precision-weighted) (threshold=0.336605) =====
Accuracy: 0.9513274336283186
Precision: 0.5915492957746479
Recall: 0.5316455696202531
F1: 0.56
ROC AUC: 0.926598138437596
Confusion Matrix:
[[1248   29]
 [  37   42]]

===== Metrics for F1 (balanced) (threshold=0.307679) =====
Accuracy: 0.950589970501475
Precision: 0.575
Recall: 0.5822784810126582
F1: 0.5786163522012578
ROC AUC: 0.926598138437596
Confusion Matrix:
[[1243   34]
 [  33   46]]

===== Metrics for F2 (recall-weighted) (threshold=0.000739) =====
Accuracy: 0.8923303834808259
Precision: 0.3397129186602871
Recall: 0.8987341772151899
F1: 0.4930555555555556
ROC AUC: 0.926598138437596
Confusion Matrix:
[[1139  138]
 [   8   71]]


## LoRA Training: Local

### Set up the LoRA Run

Low-rank adapters. Trains the attention heads and feedforwards layers by tweaking matrices- not full model training. In theory this will make the model more adapted to our specific task. Takes ~30 mins for 1000 steps, assuming it doesn't fail after 853, which has happened. 

In [162]:
from chronos.chronos2 import preprocess

ct = datetime.datetime.now()
print("current time:", ct)


# Prepare data for fine-tuning using the retail sales dataset
train_inputs_lora = preprocess.from_data_frame(
    df_context,
    target_columns=["target_conflict_indicator_local"],
    prediction_length=12,
    id_column="id",
    timestamp_column="periodStart",
    known_covariates_names=[ "name",	"country",	"latitude",	"longitude",	"minBorderDistanceKm",	"minCapitalDistanceKm", "is_voting"], #Knowable values in the future dataset. Remaining columns are treated as past-only covariates
)
print("executed train_inputs step")

# Load Chronos‑2
pipeline_lora_local = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu"   #"CUDA also an option if you're a baller with a big GPU (I am not). 
)

# Fine-tune the model with LoRA
lora_chronos_pipeline_local = pipeline_lora_local.fit(
    inputs=train_inputs_lora,
    prediction_length=12, #must match the number of observations in the future_df per place. 
    num_steps=1000,#Set to 10 just to check that it works. 
    learning_rate=1e-4,
    batch_size=32,
    logging_steps=100,
    finetune_mode="lora"
)
print("executed pipeline_lora.fit step")

joblib.dump(lora_chronos_pipeline_local, f"chronos2_model_LoRA_local.pkl") #save the full model as a pickle file. 


ct = datetime.datetime.now()
print("current time:", ct)



current time: 2026-09-27 14:33:33.077250
executed train_inputs step


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Step,Training Loss
100,0.055348
200,0.045403
300,0.055840
400,0.058434
500,0.049928
600,0.057108
700,0.052493
800,0.049018
900,0.076889
1000,0.044868


executed pipeline_lora.fit step
current time: 2026-09-27 15:00:46.940529


In [163]:
ct = datetime.datetime.now()
print("current time:", ct)

pred_df_local_lora = lora_chronos_pipeline_local.predict_df(
    df =df_context,
    future_df=df_future.drop(columns=['target_conflict_indicator_local', 'target_conflict_indicator_regional']).copy(),  #drop the target cols from the future df; we don't need them. 
    prediction_length=12,  # Number of steps to forecast 
    quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
    id_column="id",  # Column identifying different time series
    timestamp_column="periodStart",  # Column with datetime information
    target="target_conflict_indicator_local",  # Column(s) with time series values to predict
    cross_learning=True,  # Enable cross-learning;
)
pred_df_local_lora

ct = datetime.datetime.now()
print("current time:", ct)


,id,periodStart,target_name,predictions,0.1,0.5,0.9
0,Central_African_Republic_Bambari,2023-08-03,target_conflict_indicator_local,0.025143,-0.013960,0.025143,1.050816
1,Central_African_Republic_Bambari,2023-08-31,target_conflict_indicator_local,0.030658,-0.001963,0.030658,1.075720
2,Central_African_Republic_Bambari,2023-09-28,target_conflict_indicator_local,-0.010159,-0.013128,-0.010159,0.978405
3,Central_African_Republic_Bambari,2023-10-26,target_conflict_indicator_local,0.130375,0.001653,0.130375,1.078918
4,Central_African_Republic_Bambari,2023-11-23,target_conflict_indicator_local,0.023918,0.008790,0.023918,1.028021
...,...,...,...,...,...,...,...
1351,Uganda_Zombo,2024-02-15,target_conflict_indicator_local,0.000386,-0.000524,0.000386,0.002318
1352,Uganda_Zombo,2024-03-14,target_conflict_indicator_local,0.000361,-0.000295,0.000361,0.003211
1353,Uganda_Zombo,2024-04-11,target_conflict_indicator_local,0.000122,-0.000578,0.000122,0.002904
1354,Uganda_Zombo,2024-05-09,target_conflict_indicator_local,0.000352,-0.000484,0.000352,0.002359


In [169]:
eval_df_local_lora = pred_df_local_lora.merge(
    df_future[["id", "periodStart", "target_conflict_indicator_local"]],
    on=["id", "periodStart"],
    how="inner"
)
eval_df_local_lora

,id,periodStart,target_name,predictions,0.1,0.5,0.9,target_conflict_indicator_local
0,Central_African_Republic_Bambari,2023-08-03,target_conflict_indicator_local,0.025143,-0.013960,0.025143,1.050816,0
1,Central_African_Republic_Bambari,2023-08-31,target_conflict_indicator_local,0.030658,-0.001963,0.030658,1.075720,0
2,Central_African_Republic_Bambari,2023-09-28,target_conflict_indicator_local,-0.010159,-0.013128,-0.010159,0.978405,0
3,Central_African_Republic_Bambari,2023-10-26,target_conflict_indicator_local,0.130375,0.001653,0.130375,1.078918,0
4,Central_African_Republic_Bambari,2023-11-23,target_conflict_indicator_local,0.023918,0.008790,0.023918,1.028021,0
...,...,...,...,...,...,...,...,...
1351,Uganda_Zombo,2024-02-15,target_conflict_indicator_local,0.000386,-0.000524,0.000386,0.002318,0
1352,Uganda_Zombo,2024-03-14,target_conflict_indicator_local,0.000361,-0.000295,0.000361,0.003211,0
1353,Uganda_Zombo,2024-04-11,target_conflict_indicator_local,0.000122,-0.000578,0.000122,0.002904,0
1354,Uganda_Zombo,2024-05-09,target_conflict_indicator_local,0.000352,-0.000484,0.000352,0.002359,0


In [170]:


import numpy as np
from sklearn.metrics import (
    precision_recall_curve, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_auc_score
)

# ============================
# 1. Extract true labels + probabilities
# ============================

y_true = eval_df_local_lora["target_conflict_indicator_local"].values          # binary 0/1 target
y_prob = eval_df_local_lora["0.5"].values                       # Chronos2 median probability


# ============================
# 2. Compute precision-recall curve
# ============================

precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

# Align lengths: thresholds is length N-1
precision = precision[:-1]
recall = recall[:-1]


# ============================
# 3. Compute F0.5, F1, F2 scores
# ============================

# F0.5: precision weighted twice as much as recall
f05_scores = (1.25 * precision * recall) / (0.25 * precision + recall + 1e-9)

# F1: balanced
f1_scores_curve = (2 * precision * recall) / (precision + recall + 1e-9)

# F2: recall weighted twice as much as precision
f2_scores = (5 * precision * recall) / (4 * precision + recall + 1e-9)


# ============================
# 4. Select best thresholds
# ============================

best_idx_f05 = np.argmax(f05_scores)
best_threshold_f05 = thresholds[best_idx_f05]

best_idx_f1 = np.argmax(f1_scores_curve)
best_threshold_f1 = thresholds[best_idx_f1]

best_idx_f2 = np.argmax(f2_scores)
best_threshold_f2 = thresholds[best_idx_f2]


print("Best threshold (F0.5):", best_threshold_f05)
print("Best threshold (F1):", best_threshold_f1)
print("Best threshold (F2):", best_threshold_f2)


# ============================
# 5. Function to compute metrics for a given threshold
# ============================

def compute_metrics(threshold, name):
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n===== Metrics for {name} (threshold={threshold:.6f}) =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1:", f1_score(y_true, y_pred))
    print("ROC AUC:", roc_auc_score(y_true, y_prob))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


# ============================
# 6. Output metrics for each threshold
# ============================

compute_metrics(best_threshold_f05, "F0.5 (precision-weighted)")
compute_metrics(best_threshold_f1, "F1 (balanced)")
compute_metrics(best_threshold_f2, "F2 (recall-weighted)")


Best threshold (F0.5): 0.97374487
Best threshold (F1): 0.23394427
Best threshold (F2): 0.0068817437

===== Metrics for F0.5 (precision-weighted) (threshold=0.973745) =====
Accuracy: 0.8185840707964602
Precision: 0.8073654390934845
Recall: 0.6155507559395248
F1: 0.6985294117647058
ROC AUC: 0.8941781410006796
Confusion Matrix:
[[825  68]
 [178 285]]

===== Metrics for F1 (balanced) (threshold=0.233944) =====
Accuracy: 0.8193215339233039
Precision: 0.6822742474916388
Recall: 0.8812095032397408
F1: 0.7690857681432611
ROC AUC: 0.8941781410006796
Confusion Matrix:
[[703 190]
 [ 55 408]]

===== Metrics for F2 (recall-weighted) (threshold=0.006882) =====
Accuracy: 0.7404129793510325
Precision: 0.5705209656925032
Recall: 0.9697624190064795
F1: 0.7184
ROC AUC: 0.8941781410006796
Confusion Matrix:
[[555 338]
 [ 14 449]]


### Lora Training: Regional

In [171]:
from chronos.chronos2 import preprocess

ct = datetime.datetime.now()
print("current time:", ct)


# Prepare data for fine-tuning using the retail sales dataset
train_inputs_lora = preprocess.from_data_frame(
    df_context,
    target_columns=["target_conflict_indicator_regional"],
    prediction_length=12,
    id_column="id",
    timestamp_column="periodStart",
    known_covariates_names=[ "name",	"country",	"latitude",	"longitude",	"minBorderDistanceKm",	"minCapitalDistanceKm", "is_voting"], #Knowable values in the future dataset. Remaining columns are treated as past-only covariates
)
print("executed train_inputs step")

# Load Chronos‑2
pipeline_lora_regional = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu"   #"CUDA also an option if you're a baller with a big GPU (I am not). 
)

# Fine-tune the model with LoRA
lora_chronos_pipeline_regional = pipeline_lora_regional.fit(
    inputs=train_inputs_lora,
    prediction_length=12, #must match the number of observations in the future_df per place. 
    num_steps=1000,#Set to 10 just to check that it works. 
    learning_rate=1e-4,
    batch_size=32,
    logging_steps=100,
    finetune_mode="lora"
)
print("executed pipeline_lora.fit step")

joblib.dump(lora_chronos_pipeline_regional, f"chronos2_model_LoRA_regional.pkl") #save the full model as a pickle file. 


ct = datetime.datetime.now()
print("current time:", ct)



current time: 2026-09-27 15:14:30.811942
executed train_inputs step


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Step,Training Loss
100,0.015256
200,0.018504
300,0.020802
400,0.023583
500,0.019935
600,0.012781
700,0.008354
800,0.025422
900,0.033568
1000,0.017062


executed pipeline_lora.fit step
current time: 2026-09-27 15:40:52.201157


In [172]:
ct = datetime.datetime.now()
print("current time:", ct)

pred_df_regional_lora = lora_chronos_pipeline_regional.predict_df(
    df =df_context,
    future_df=df_future.drop(columns=['target_conflict_indicator_local', 'target_conflict_indicator_regional']).copy(),  #drop the target cols from the future df; we don't need them. 
    prediction_length=12,  # Number of steps to forecast 
    quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
    id_column="id",  # Column identifying different time series
    timestamp_column="periodStart",  # Column with datetime information
    target="target_conflict_indicator_regional",  # Column(s) with time series values to predict
    cross_learning=True,  # Enable cross-learning;
)
pred_df_regional_lora

ct = datetime.datetime.now()
print("current time:", ct)


current time: 2026-09-27 15:40:52.257713
current time: 2026-09-27 15:43:37.095381


In [175]:
eval_df_regional_lora = pred_df_regional_lora.merge(
    df_future[["id", "periodStart", "target_conflict_indicator_regional"]],
    on=["id", "periodStart"],
    how="inner"
)
eval_df_regional_lora

,id,periodStart,target_name,predictions,0.1,0.5,0.9,target_conflict_indicator_regional
0,Central_African_Republic_Bambari,2023-08-03,target_conflict_indicator_regional,0.015425,-0.007023,0.015425,1.072606,0
1,Central_African_Republic_Bambari,2023-08-31,target_conflict_indicator_regional,0.011579,-0.006567,0.011579,1.058636,0
2,Central_African_Republic_Bambari,2023-09-28,target_conflict_indicator_regional,-0.005263,-0.009925,-0.005263,0.954178,0
3,Central_African_Republic_Bambari,2023-10-26,target_conflict_indicator_regional,0.060956,-0.008511,0.060956,1.079874,0
4,Central_African_Republic_Bambari,2023-11-23,target_conflict_indicator_regional,-0.002016,-0.005969,-0.002016,1.094314,0
...,...,...,...,...,...,...,...,...
1351,Uganda_Zombo,2024-02-15,target_conflict_indicator_regional,0.000151,-0.000738,0.000151,0.001421,0
1352,Uganda_Zombo,2024-03-14,target_conflict_indicator_regional,0.000359,-0.000331,0.000359,0.001688,0
1353,Uganda_Zombo,2024-04-11,target_conflict_indicator_regional,0.000103,-0.000568,0.000103,0.001647,0
1354,Uganda_Zombo,2024-05-09,target_conflict_indicator_regional,0.000366,-0.000409,0.000366,0.001283,0


In [176]:


import numpy as np
from sklearn.metrics import (
    precision_recall_curve, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_auc_score
)

# ============================
# 1. Extract true labels + probabilities
# ============================

y_true = eval_df_regional_lora["target_conflict_indicator_regional"].values          # binary 0/1 target
y_prob = eval_df_regional_lora["0.5"].values                       # Chronos2 median probability


# ============================
# 2. Compute precision-recall curve
# ============================

precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

# Align lengths: thresholds is length N-1
precision = precision[:-1]
recall = recall[:-1]


# ============================
# 3. Compute F0.5, F1, F2 scores
# ============================

# F0.5: precision weighted twice as much as recall
f05_scores = (1.25 * precision * recall) / (0.25 * precision + recall + 1e-9)

# F1: balanced
f1_scores_curve = (2 * precision * recall) / (precision + recall + 1e-9)

# F2: recall weighted twice as much as precision
f2_scores = (5 * precision * recall) / (4 * precision + recall + 1e-9)


# ============================
# 4. Select best thresholds
# ============================

best_idx_f05 = np.argmax(f05_scores)
best_threshold_f05 = thresholds[best_idx_f05]

best_idx_f1 = np.argmax(f1_scores_curve)
best_threshold_f1 = thresholds[best_idx_f1]

best_idx_f2 = np.argmax(f2_scores)
best_threshold_f2 = thresholds[best_idx_f2]


print("Best threshold (F0.5):", best_threshold_f05)
print("Best threshold (F1):", best_threshold_f1)
print("Best threshold (F2):", best_threshold_f2)


# ============================
# 5. Function to compute metrics for a given threshold
# ============================

def compute_metrics(threshold, name):
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n===== Metrics for {name} (threshold={threshold:.6f}) =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1:", f1_score(y_true, y_pred))
    print("ROC AUC:", roc_auc_score(y_true, y_prob))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


# ============================
# 6. Output metrics for each threshold
# ============================

compute_metrics(best_threshold_f05, "F0.5 (precision-weighted)")
compute_metrics(best_threshold_f1, "F1 (balanced)")
compute_metrics(best_threshold_f2, "F2 (recall-weighted)")


Best threshold (F0.5): 0.22056076
Best threshold (F1): 0.18400562
Best threshold (F2): 0.0009660125

===== Metrics for F0.5 (precision-weighted) (threshold=0.220561) =====
Accuracy: 0.9513274336283186
Precision: 0.5844155844155844
Recall: 0.569620253164557
F1: 0.5769230769230769
ROC AUC: 0.8954333237512763
Confusion Matrix:
[[1245   32]
 [  34   45]]

===== Metrics for F1 (balanced) (threshold=0.184006) =====
Accuracy: 0.9498525073746312
Precision: 0.5662650602409639
Recall: 0.5949367088607594
F1: 0.5802469135802469
ROC AUC: 0.8954333237512763
Confusion Matrix:
[[1241   36]
 [  32   47]]

===== Metrics for F2 (recall-weighted) (threshold=0.000966) =====
Accuracy: 0.890117994100295
Precision: 0.33490566037735847
Recall: 0.8987341772151899
F1: 0.4879725085910653
ROC AUC: 0.8954333237512763
Confusion Matrix:
[[1136  141]
 [   8   71]]


## Save LoRA models

In [185]:
lora_chronos_pipeline_local.save_pretrained("lora_chronos_pipeline_local")
lora_chronos_pipeline_regional.save_pretrained("lora_chronos_pipeline_regional")